# IssueFix-RL — SFT Training on Kaggle 2× T4

**Upload this notebook to Kaggle, enable 2× GPU accelerator, turn on Internet, then Save & Run All.**

Pipeline:
1. Clone repo
2. Install deps
3. Run training — auto-distributes across both T4s via `notebook_launcher`
4. Zip best checkpoint → `/kaggle/working/`
5. Push to Kaggle Models page

Best checkpoint + logs persist as a Kaggle notebook version (visible in Output tab).

---
**Before running:**
- Attach your training data as a Kaggle dataset (jsonl with `prompt`/`response` keys)
- Add your wandb API key as a Kaggle secret named `WANDB_API_KEY` (optional)
- Set `DATA_PATH` in cell 2 to your mounted dataset path

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/ramprasathk07/IssueFix-RL.git"
DATA_PATH = "/kaggle/input/your-dataset-name/opencode_sft_filtered.jsonl"  # <-- set your dataset path
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import os, subprocess, sys

REPO_DIR = "/kaggle/working/IssueFix-RL"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("Repo:", os.getcwd())

# Confirm 2 GPUs
result = subprocess.run(["nvidia-smi", "--list-gpus"], capture_output=True, text=True)
gpu_lines = [l for l in result.stdout.strip().splitlines() if l.strip()]
print(f"GPUs available: {len(gpu_lines)}")
for g in gpu_lines:
    print(" ", g)

In [ ]:
# torch pre-installed on Kaggle — installs everything else
!pip install -q -r requirements.txt

In [ ]:
# Pull wandb key from Kaggle Secrets (set via Add-ons → Secrets in the notebook editor)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_key
    print("wandb key loaded from Kaggle Secrets")
except Exception:
    print("No WANDB_API_KEY secret found — wandb logging disabled")
    os.environ["WANDB_DISABLED"] = "true"

In [ ]:
import json
from pathlib import Path

data_file = Path(DATA_PATH)
assert data_file.exists(), (
    f"Data not found at {DATA_PATH}.\n"
    "Attach your dataset in the notebook editor: Data → Add Data, "
    "then update DATA_PATH in cell 2."
)

with open(data_file) as f:
    lines = f.readlines()
sample = json.loads(lines[0])
print(f"Examples : {len(lines)}")
print(f"Keys     : {list(sample.keys())}")
print(f"Prompt   : {sample.get('prompt','')[:100]}")
print(f"Response : {sample.get('response','')[:100]}")

In [ ]:
import yaml

with open("configs/sft.yaml") as f:
    cfg = yaml.safe_load(f)

tp = cfg["training_params"]
dp = cfg["dataloader_params"]
mp = cfg["model_params"]

eff = dp["batch_size"] * tp["num_gpus"] * tp["gradient_accumulation_steps"]
print(f"Model           : {mp['base_model']}")
print(f"Epochs          : {tp['num_epochs']}")
print(f"LR              : {tp['learning_rate']}")
print(f"GPUs            : {tp['num_gpus']}")
print(f"Per-GPU batch   : {dp['batch_size']}")
print(f"Grad accum      : {tp['gradient_accumulation_steps']}")
print(f"Effective batch : {eff}")
print(f"Max length      : {dp['max_length']}")
print(f"bf16            : {tp['bf16']}")
print(f"grad_ckpt       : {tp['gradient_checkpointing']}")
print(f"Output dir      : {tp['output_dir']}")
print(f"wandb project   : {tp.get('wandb_project')}")
print(f"Kaggle push     : {tp.get('push_to_kaggle')} → {tp.get('kaggle_model_handle')}")

In [ ]:
# ── TRAINING ──────────────────────────────────────────────────────────────────
# notebook_launcher (called inside train.py) spawns both T4 processes.
# After all epochs: best checkpoint zipped + pushed to Kaggle Models.
# This cell is blocking — use Save Version → Save & Run All to run in background.

!python train.py --config configs/sft.yaml --data {DATA_PATH}

In [ ]:
# ── RESUME FROM CHECKPOINT (skip if running fresh) ────────────────────────────
# Uncomment, set CKPT, and run this cell instead of the training cell above.

# CKPT = "outputs/sft_run1/checkpoint-epoch1-step150"
# !python train.py --config configs/sft.yaml --data {DATA_PATH} --resume {CKPT}

In [ ]:
# ── OUTPUT SUMMARY ────────────────────────────────────────────────────────────
from pathlib import Path
import yaml

with open("configs/sft.yaml") as f:
    out_dir = Path(yaml.safe_load(f)["training_params"]["output_dir"])

checkpoints = sorted(out_dir.glob("checkpoint-*")) if out_dir.exists() else []
print(f"Checkpoints saved: {len(checkpoints)}")
for c in checkpoints:
    files = [f.name for f in c.iterdir()]
    print(f"  {c.name}: {files}")

print()
zips = list(Path("/kaggle/working").glob("*.zip"))
print(f"Zipped checkpoints ({len(zips)}):")
for z in zips:
    print(f"  {z.name}  {z.stat().st_size / 1e6:.1f} MB")

In [ ]:
# ── CHECKPOINT TESTS + SAMPLE GENERATIONS (optional) ─────────────────────────
checkpoints = sorted(Path(out_dir).glob("checkpoint-*")) if out_dir.exists() else []
if checkpoints:
    latest = checkpoints[-1]
    print(f"Testing checkpoint: {latest}")
    !pytest src/tests/test_checkpoint.py -v -s --ckpt {latest}
else:
    print("No checkpoint found — training may have failed")